### NAME: Bangalore AQI + Weather Analytics — Data Cleaning


## Task: Data Cleaning
### Overview
This notebook loads the raw AQI station snapshots and weather snapshots collected by `fetch_data.py`, cleans and merges them into a single analysis-ready dataset (`cleaned_bangalore_aqi.csv`).

### Step 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


### Step 2: Load the Datasets
Two raw files come out of `fetch_data.py`:
- `data/aqi_stations.csv` — one row per station per fetch 
- `data/weather.csv` — one row per fetch 

In [2]:
def load_dataset(file_path):
    """Load the dataset from a file."""
    try:
        return pd.read_csv(file_path)
    except Exception as error:
        print(f"Error loading dataset: {error}")
        return None

aqi = load_dataset("data/aqi_stations.csv")
weather = load_dataset("data/weather.csv")
print(aqi.shape, weather.shape)
aqi.head()


(392, 6) (66, 9)


,fetched_at,station_uid,station_name,latitude,longitude,aqi
0,2026-09-15T00:00:00+00:00,8686,"City Railway Station, Bangalore",12.9767,77.5713,NaN
1,2026-09-15T00:00:00+00:00,8191,"BWSSB, Bangalore",12.9634,77.5855,88.0
2,2026-09-15T00:00:00+00:00,8190,"BTM, Bangalore",12.9166,77.6101,103.7
3,2026-09-15T00:00:00+00:00,3758,"Peenya, Bangalore",13.0280,77.5200,133.6
4,2026-09-15T00:00:00+00:00,12441,"BWSSB Kadabesanahalli, Bengaluru",12.9370,77.6910,99.4


### Step 3: Parse Timestamps
Convert `fetched_at` to a proper datetime, and derive `date`/`hour` columns we'll use throughout the EDA (same role as `last_update` in the original).

In [3]:
def convert_column_types(dataframe):
    """Convert timestamp columns and derive date/hour helper columns."""
    dataframe = dataframe.copy()
    dataframe['fetched_at'] = pd.to_datetime(
    dataframe['fetched_at'],
    format='mixed'
)
    dataframe['date'] = dataframe['fetched_at'].dt.date
    dataframe['hour'] = dataframe['fetched_at'].dt.hour
    return dataframe

aqi = convert_column_types(aqi)
weather = convert_column_types(weather)
aqi.head()


,fetched_at,station_uid,station_name,latitude,longitude,aqi,date,hour
0,2026-09-15 00:00:00+00:00,8686,"City Railway Station, Bangalore",12.9767,77.5713,NaN,2026-09-15,0
1,2026-09-15 00:00:00+00:00,8191,"BWSSB, Bangalore",12.9634,77.5855,88.0,2026-09-15,0
2,2026-09-15 00:00:00+00:00,8190,"BTM, Bangalore",12.9166,77.6101,103.7,2026-09-15,0
3,2026-09-15 00:00:00+00:00,3758,"Peenya, Bangalore",13.0280,77.5200,133.6,2026-09-15,0
4,2026-09-15 00:00:00+00:00,12441,"BWSSB Kadabesanahalli, Bengaluru",12.9370,77.6910,99.4,2026-09-15,0


### Step 4: Handle Missing / Offline Stations
WAQI stations occasionally report `aqi` as missing when a sensor is temporarily offline. We check for and report these rather than silently dropping them, since "how often is a station offline" is itself a useful signal.

In [4]:
def check_null_values(dataframe, label):
    null_counts = dataframe.isnull().sum()
    print(f"\nNull values in {label}:")
    print(null_counts[null_counts > 0])
    return null_counts

def check_duplicate_values(dataframe, label):
    duplicate_count = dataframe.duplicated().sum()
    print(f"Duplicate rows in {label}: {duplicate_count}")
    return duplicate_count

_ = check_null_values(aqi, "aqi")
_ = check_null_values(weather, "weather")
_ = check_duplicate_values(aqi, "aqi")
_ = check_duplicate_values(weather, "weather")

# Flag offline readings rather than dropping — useful for a "station reliability" metric later
aqi['is_offline'] = aqi['aqi'].isnull()



Null values in aqi:
aqi    15
dtype: int64

Null values in weather:
Series([], dtype: int64)
Duplicate rows in aqi: 0
Duplicate rows in weather: 0


### Step 5: Merge AQI + Weather on Timestamp
Both files are fetched in the same run, so `fetched_at` lines up between them (small tolerance in case one call succeeded and the other lagged by a few seconds). We use `merge_asof` to align each AQI snapshot with the nearest weather reading, mirroring how the original project merged bike + weather data on timestamp.

In [5]:
aqi_sorted = aqi.sort_values('fetched_at')
weather_sorted = weather.sort_values('fetched_at')

merged = pd.merge_asof(
    aqi_sorted,
    weather_sorted[['fetched_at', 'temperature_c', 'humidity_pct', 'pressure_hpa',
                     'wind_speed_ms', 'rain_1h_mm', 'clouds_pct', 'weather_main', 'weather_desc']],
    on='fetched_at',
    direction='nearest',
    tolerance=pd.Timedelta('15min')
)
merged.head()


,fetched_at,station_uid,station_name,latitude,longitude,aqi,date,hour,is_offline,temperature_c,humidity_pct,pressure_hpa,wind_speed_ms,rain_1h_mm,clouds_pct,weather_main,weather_desc
0,2026-09-15 00:00:00+00:00,8686,"City Railway Station, Bangalore",12.9767,77.5713,NaN,2026-09-15,0,True,18.2,77.1,1011.2,2.27,0.0,57.7,Clouds,scattered clouds
1,2026-09-15 00:00:00+00:00,8191,"BWSSB, Bangalore",12.9634,77.5855,88.0,2026-09-15,0,False,18.2,77.1,1011.2,2.27,0.0,57.7,Clouds,scattered clouds
2,2026-09-15 00:00:00+00:00,8190,"BTM, Bangalore",12.9166,77.6101,103.7,2026-09-15,0,False,18.2,77.1,1011.2,2.27,0.0,57.7,Clouds,scattered clouds
3,2026-09-15 00:00:00+00:00,3758,"Peenya, Bangalore",13.0280,77.5200,133.6,2026-09-15,0,False,18.2,77.1,1011.2,2.27,0.0,57.7,Clouds,scattered clouds
4,2026-09-15 00:00:00+00:00,12441,"BWSSB Kadabesanahalli, Bengaluru",12.9370,77.6910,99.4,2026-09-15,0,False,18.2,77.1,1011.2,2.27,0.0,57.7,Clouds,scattered clouds


### Step 6: Save Cleaned Dataset

In [6]:
merged.to_csv('data/cleaned_bangalore_aqi.csv', index=False)
print(f"Saved {len(merged)} rows to data/cleaned_bangalore_aqi.csv")


Saved 392 rows to data/cleaned_bangalore_aqi.csv


### Data Cleaning Completed
The merged, cleaned dataset is ready for analysis in `02_eda_analysis.ipynb`.